# Structure development

This notebook will be used to define the data structures in a test-like environment.

## The goal

In the end, I want an OOP data structure with a result such as:

```
schedule.[shift-segment].[location-name] = [...employee]
```

There should be a simple way of looping through the shift segments and locations:

```
for segment in schedule.shift_segments:
    for location in segment.locations:
        print(location.employees) # should output [...employee]
```

Let's start from the top, the `schedule` class.

In [ ]:
schedule = {
    (9, 11): {
        "service-pt-1": ["chris"],
        "service-pt-2": ["carla"],
        "pickup-window": ["jess"],
    },
    (11, 13): {
        "service-pt-1": ["carla"],
        "service-pt-2": ["jess"],
        "pickup-window": ["chris"],
    },
    (13, 14): {
        "service-pt-1": ["chris"],
        "service-pt-2": ["carla"],
        "pickup-window": ["jess"],
    },
    (14, 16): {
        "service-pt-1": ["carla"],
        "service-pt-2": ["jess"],
        "pickup-window": ["chris"],
    },
}

location_names = ["service-pt-1", "service-pt-2", "pickup-window"]

for slot, locations in schedule.items():
    t1, t2 = slot
    if t1 > 12:
        t1 -= 12
    if t2 > 12:
        t2 -= 12

    print(f"{t1}-{t2}")
    for location, employees in locations.items():
        print(f"  {location}: {', '.join(employees)}")

9-11
  service-pt-1: chris
  service-pt-2: carla
  pickup-window: jess
11-1
  service-pt-1: carla
  service-pt-2: jess
  pickup-window: chris
1-2
  service-pt-1: chris
  service-pt-2: carla
  pickup-window: jess
2-4
  service-pt-1: carla
  service-pt-2: jess
  pickup-window: chris


In [ ]:
from dataclasses import dataclass, field


@dataclass
class Location:
    name: str
    required_roles: list[str] = field(default_factory=list)
    employees: list[str] = field(default_factory=list)
    min_staff: int = 1

    def add_employee(self, name: str):
        if name not in self.employees:
            self.employees.append(name)

    def remove_employee(self, name: str):
        if name in self.employees:
            self.employees.remove(name)

    def is_empty(self) -> bool:
        return len(self.employees) == 0

    def needs_staff(self) -> bool:
        return len(self.employees) < self.min_staff

@dataclass
class TimeSlot:
    start_time: int
    end_time: int
    locations: dict[str, Location] = field(default_factory=dict)

    def get_time_label(self) -> str:
        t1 = self.start_time - 12 if self.start_time > 12 else self.start_time
        t2 = self.end_time - 12 if self.end_time > 12 else self.end_time
        return f"{t1}-{t2}"

schedule: list[TimeSlot] = [
    TimeSlot(9, 11, {
        "service-pt-1": Location("service-pt-1", required_roles=["manager"]),
        "service-pt-2": Location("service-pt-2", min_staff=2),
        "pickup-window": Location("pickup-window"),
    }),
    TimeSlot(11, 13, {
        "service-pt-1": Location("service-pt-1", required_roles=["manager"]),
        "service-pt-2": Location("service-pt-2", min_staff=2),
        "pickup-window": Location("pickup-window"),
    }),
]

schedule[0].locations["service-pt-1"].add_employee("chris")
schedule[0].locations["pickup-window"].add_employee("jess")

for slot in schedule:
    print(f"\nShift: {slot.get_time_label()}")

    for loc_name, location in slot.locations.items():
        if location.is_empty():
            print(f"  [WARNING] {loc_name} is empty! Requires {location.min_staff} staff.")
        elif location.needs_staff():
            print(f"  [WARNING] {loc_name} is understaffed! Has {len(location.employees)}/{location.min_staff}.")
        else:
            print(f"  {loc_name}: {', '.join(location.employees)}")



Shift: 9-11
  service-pt-1: chris
  [WARNING] service-pt-2 is empty! Requires 2 staff.
  pickup-window: jess

Shift: 11-1
  [WARNING] service-pt-1 is empty! Requires 1 staff.
  [WARNING] service-pt-2 is empty! Requires 2 staff.
  [WARNING] pickup-window is empty! Requires 1 staff.


Let's try and use the models for data creation.

In [ ]:
import src.models as md

terry = md.Employee(name="Terry Folds", position="Manager", experience=143)
pat = md.Employee(name="Pat Smith", position="Assistant Manager", experience=110)
harold = md.Employee(name="Harold Engels", position="Supervisor", experience=98)
robert = md.Employee(name="Robert Young", position="Supervisor", experience=98)
sally = md.Employee(name="Sally McQueen", position="Supervisor", experience=99)
mary = md.Employee(name="Mary Whitaker", position="Full Time", experience=49)
chris = md.Employee(name="Chris Wright", position="Full Time", experience=84)
frank = md.Employee(name="Frank Holmes", position="Full Time", experience=67)
bianca = md.Employee(name="Bianca Carson", position="Full Time", experience=42)
james = md.Employee(name="James Cartwright", position="Part Time", experience=24)
ronaldo = md.Employee(name="Ronaldo Vasquez", position="Part Time", experience=22)
tripp = md.Employee(name="Tripp Jones", position="Security Full Time", experience=20)
louis = md.Employee(name="Louis Stockton", position="Security Part Time", experience=12)
veronica = md.Employee(name="Veronica Saturn", position="Shelver", experience=9)
larry = md.Employee(name="Larry Jones-Smith", position="Shelver", experience=2)

employees = [
    terry,
    pat,
    harold,
    robert,
    sally,
    mary,
    chris,
    frank,
    bianca,
    james,
    ronaldo,
    tripp,
    louis,
    veronica,
    larry
]

for employee in employees:
    print(employee.name)
    print(f"  {employee.initials}")
    print(f"  {employee.sharepoint}")

Terry
  TF
  TFolds
Pat
  PS
  PSmith
Harold
  HE
  HEngels
Robert
  RY
  RYoung
Sally
  SM
  SMcQueen
Mary
  MW
  MWhitaker
Chris
  CW
  CWright
Frank
  FH
  FHolmes
Bianca
  BC
  BCarson
James
  JC
  JCartwright
Ronaldo
  RV
  RVasquez
Tripp
  TJ
  TJones
Louis
  LS
  LStockton
Veronica
  VS
  VSaturn
Larry
  LJS
  LJones-Smith
